# Train the remote-sensing captioner, second round (VRSBench + NWPU-Captions)

**Goal:** given ONE satellite/aerial image, write a human-readable description of it -- the "explain / describe this image" capability the app was missing (its VQA specialist answers in one or two words by construction, and its object detector only knows a handful of object types).

**Why a second round.** The first run (`kaggle_finetune_caption_vrsbench.ipynb`, VRSBench only) scored well on VRSBench's held-out images (BLEU-4 0.022 -> 0.118, CIDEr 0.001 -> 0.300, and it reproduced the reference length: 46 words). But read against 21 real Esri scenes whose contents are known, only about half of its descriptions were right: Iowa farmland became "a dense urban area with a ground track field" (with the *highest* confidence of the run), a tennis complex "an urban roundabout", a wind farm "a winding river and small vehicles". The cause is the training data, not the input scale (four scale variants gave the same answers): VRSBench's captions are generated from DOTA/DIOR **object** annotations, so the model describes everything in those object classes (ground track field, roundabout, harbour, small vehicle, bridge) and has never seen farmland, forest, desert, a lake or a power station as a *subject*. More VRSBench cannot fix that; a dataset whose captions are about the scene can.

**Data -- two sources, two prompts (modes), so neither drowns the other:**

| mode | prompt | source | what the captions are |
|---|---|---|---|
| detailed | `Describe the image in detail.` | [VRSBench](https://huggingface.co/datasets/xiang709/VRSBench) (CC-BY-4.0), unchanged from round one | ~46-word object-centric paragraphs |
| brief | `Briefly describe the scene.` | [NWPU-Captions](https://github.com/HaiyanHuang98/NWPU-Captions) (captions, Cheng et al. 2022) on the [NWPU-RESISC45](https://huggingface.co/datasets/jonathan-roberts1/NWPU-RESISC45) images | one-sentence scene descriptions (mean 12 words) covering 45 scene classes -- farmland (circular and rectangular), forest, desert, lake, mountain, meadow, tennis court, stadium, golf course, harbour, thermal power station, airport, ... |

Both are research-use datasets; nothing is redistributed (the trained weights stay in the private project).

**Model:** [SmolVLM-500M-Instruct](https://huggingface.co/HuggingFaceTB/SmolVLM-500M-Instruct) (Apache-2.0, ungated), LoRA on the language model plus a trainable vision-to-text connector, exactly as in round one -- it has to fit next to the other specialists on a 16GB laptop.

## What was checked before writing this (so the cells below aren't guesses)

* **From round one, unchanged:** the processor's default image splitting makes 1,088 image tokens vs 64 with it off (splitting stays off); VRSBench's captions carry provenance boilerplate ("sourced from GoogleEarth", "captured by GF with medium resolution") that would teach the model to hallucinate a source and resolution, so it is stripped from targets and references; no geometric augmentation (captions say "on the left"); loss on the answer only.
* **NWPU-Captions' file** (`dataset_nwpu.json`, 20.5MB, plain GitHub raw -- the image tarball next to it is Git-LFS and not used): 31,500 images = 45 classes x 700, the authors' own split 25,200 train / 3,150 val / 3,150 test, five human captions per image, mean 12.3 words (min 1, max 51). Captions have a space before their punctuation ("... on the open area ."), some start lowercase and a few are one word, so they are normalised and filtered (4-60 words, ASCII).
* **The images join by filename.** The Hugging Face mirror of NWPU-RESISC45 keeps the original names (`airplane_001.jpg`) in its parquet, the same keys the captions use; the notebook asserts that at least 99.5% of the mirror's images have usable captions rather than assuming it. (Hugging Face's CDN is fast from Kaggle; GitHub LFS is a fragile source.)
* **The processor resizes every image to 512x512** -- checked: a 256px NWPU image is upsampled (its attention mask covers the whole tensor, so there is no black padding to learn from), a 1024x822 capture is squashed to a square -- so NWPU's small images take exactly the same 64-token path as VRSBench's and as the app's captures.

## Design

* **Batches are homogeneous in mode.** A batch of one-sentence captions padded to a paragraph's length wastes compute, and it lets one mode's token count dominate a batch's loss. Each epoch, batches are drawn per mode and then shuffled together.
* **Each epoch every image is seen once**; an NWPU image gets one of its five captions at random (a different one most epochs).
* **Model selection uses both modes:** validation loss is measured separately on 300 held-out VRSBench images and 315 held-out NWPU images (7 per class), and the weights with the lowest *mean* of the two are kept.
* **No zero-shot re-run.** Round one measured the base model on these exact 1,000 VRSBench test images (deterministic selection, same cleaning); those numbers are recorded below and the new detailed-mode scores are compared against both them and round one's fine-tuned scores, so a regression in the detailed mode would show.
* **Brief-mode evaluation** uses 30 test images per scene class (1,350, stratified, from the authors' test split) with all five references each, plus a "does it name its own scene" check: for each class, the words its training captions use far more than other classes' do ("farmland", "runway", "tennis", ...), and how often the generated caption uses one of them -- compared with how often a human reference caption does.
* The real acceptance test is not on Kaggle: it is the 21 known real scenes, read by a person after the checkpoint is downloaded.

Run cells top to bottom. Kaggle: enable **Internet** and a **GPU**.

## 0. GPU compatibility check

**Found live, via an actual failed run on Kaggle**: this session's preinstalled PyTorch build
(2.10.0+cu128) only supports CUDA compute capabilities sm_70 and up -- it silently drops support
for the Pascal-generation **P100** (sm_60), one of the two GPU types Kaggle itself still offers
here (T4x2 or P100, per the note below). Landing on a P100 crashed training ~40 seconds in with
`CUDA error: no kernel image is available for execution on the device` -- a real run, not a
hypothetical. The cell below detects the actual GPU via `nvidia-smi` (no torch import needed yet,
so this runs before torch's own compute-capability list is fixed for the process) and reinstalls
a CUDA 11.8 build if the assigned GPU isn't in the preinstalled build's supported list -- CUDA 11.8
wheels cover Pascal through Hopper, so this works regardless of which GPU Kaggle happens to assign.

In [ ]:
import subprocess, sys

try:
    cc_raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc_raw.split(".")
    needed_sm = f"sm_{major}{minor}"
except Exception as e:
    needed_sm = None
    print(f"Could not query GPU compute capability via nvidia-smi ({e}) -- skipping the compatibility check.")

if needed_sm:
    # Check the INSTALLED build's supported architectures in a SEPARATE PROCESS, not an in-process
    # `import torch` -- Python caches imports in sys.modules, so even an aliased/deleted in-process
    # import here would make a LATER `import torch` in the next cell silently return the stale
    # cached module instead of a fresh one. Confirmed live: an earlier version of this cell did
    # `import torch as _torch_probe`, and the kernel died ~80s after reinstalling -- the old torch
    # stayed resident in this process while its .so files got replaced out from under it on disk.
    check = subprocess.run(
        [sys.executable, "-c",
         "import torch; print(' '.join(torch.cuda.get_arch_list()) if torch.cuda.is_available() else '')"],
        capture_output=True, text=True,
    )
    supported = check.stdout.split()
    if needed_sm not in supported:
        print(f"GPU needs {needed_sm}, not in the preinstalled torch build's supported list "
              f"{supported} -- reinstalling a CUDA 11.8 build (covers Pascal through Hopper)...")
        # Uninstall the whole torch/torchvision/torchaudio trio first, then install all three
        # together from the SAME cu118 index in one resolution -- reinstalling `torch` alone
        # left mismatched torchvision/nccl versions behind, which crashed two live runs with
        # unrelated-looking errors (undefined symbol ncclCommShrink; aten.OpaqueObject not
        # registered) that were actually both this same root cause from a different angle.
        subprocess.run(["pip", "uninstall", "-y", "-q", "torch", "torchvision", "torchaudio"], check=True)
        subprocess.run(
            ["pip", "install", "-q", "torch", "torchvision", "torchaudio",
             "--index-url", "https://download.pytorch.org/whl/cu118"],
            check=True,
        )
        print("Reinstalled. The check above ran in a subprocess, so THIS process has never "
              "imported torch itself -- the next cell's `import torch` will be a genuinely fresh "
              "import, not a cached one.")
    else:
        print(f"GPU compute capability {needed_sm} already supported by the preinstalled build.")

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))

## 1. Setup

In [ ]:
!pip install -q peft pycocoevalcap hf_transfer
# Kaggle's image ships an old torchao (0.10) that the freshly installed peft refuses to import alongside
# ("Found an incompatible version of torchao ... only versions above 0.16.0") -- round one's first run died
# on exactly this, at get_peft_model, after 20 minutes of downloads. Nothing here uses torchao.
!pip uninstall -y -q torchao

In [ ]:
import os, sys, json, re, math, time, random, zipfile, io, shutil, subprocess, collections
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # set before huggingface_hub is imported (see the download cell)

# Local smoke-test hooks -- never set on Kaggle. CAP_DATA_ROOT points at a folder holding the four
# VRSBench files (real captions, a handful of real images in mini zips) plus nwpu_mini.parquet (a few
# hundred real NWPU images, all 45 classes) and dataset_nwpu.json, so the whole notebook can be run end
# to end on a laptop before any GPU time is spent.
SMOKE_TEST = bool(os.environ.get("CAP_SMOKE_TEST"))
DATA_ROOT = os.environ.get("CAP_DATA_ROOT")
MODEL_ID = os.environ.get("CAP_MODEL_ID", "HuggingFaceTB/SmolVLM-500M-Instruct")

# Two modes, told apart by the prompt alone. "detailed" is round one's prompt, word for word.
PROMPTS = {"detailed": "Describe the image in detail.", "brief": "Briefly describe the scene."}
MODE = {"vrs_train": "detailed", "vrs_eval": "detailed", "nwpu": "brief"}  # caption source -> mode
SEED = 0
N_VAL, N_TEST = (8, 8) if SMOKE_TEST else (300, 1000)  # VRSBench (detailed): validation / test images
PER_CLASS_VAL, PER_CLASS_TEST = (1, 1) if SMOKE_TEST else (7, 30)  # NWPU (brief): per scene class, 45 classes
EPOCHS = 1 if SMOKE_TEST else 2
TARGET_BATCH = 2 if SMOKE_TEST else 32  # the EFFECTIVE batch size
BATCH = 1 if SMOKE_TEST else TARGET_BATCH  # the micro-batch: the pre-flight lowers it to what the GPU can hold
ACCUM = TARGET_BATCH // BATCH  # gradient-accumulation steps that make up the difference (smoke mode uses 2 to exercise this path)
WORKERS = 0 if SMOKE_TEST else 4
LR = 2e-4
MAX_NEW_TOKENS = {"detailed": 24 if SMOKE_TEST else 128, "brief": 16 if SMOKE_TEST else 64}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| smoke test:", SMOKE_TEST, "| model:", MODEL_ID)

## 2. Model input, generation and metrics

`chat()` builds exactly the prompt the app will use at inference, for either mode. `collate` masks everything up to and including the last `Assistant:` marker, so only the caption (and the end-of-utterance token) is trained on. These helpers need no data, so they come first -- the pre-flight below runs them before anything is downloaded.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.do_image_splitting = False  # 64 image tokens per 512px image, not 1,088
tokenizer = processor.tokenizer
ASSISTANT_IDS = tokenizer.encode("Assistant:", add_special_tokens=False)

def chat(caption=None, mode="detailed"):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPTS[mode]}]}]
    if caption is None:
        return processor.apply_chat_template(messages, add_generation_prompt=True)
    messages.append({"role": "assistant", "content": [{"type": "text", "text": caption}]})
    return processor.apply_chat_template(messages, add_generation_prompt=False).rstrip("\n")  # ends on <end_of_utterance>

class ZipImages:
    """Reads images straight out of a zip. The handle is opened lazily, per process, so DataLoader
    workers each get their own."""
    def __init__(self, path, folder):
        self.path, self.folder, self._zip = path, folder, None
    def load(self, name):
        if self._zip is None:
            self._zip = zipfile.ZipFile(self.path)
        return Image.open(io.BytesIO(self._zip.read(f"{self.folder}/{name}"))).convert("RGB")
    def __getstate__(self):
        state = self.__dict__.copy(); state["_zip"] = None
        return state

class ParquetImages:
    """The NWPU-RESISC45 images by filename, held as JPEG bytes in memory (~430MB for all 31,500) and
    decoded on demand. DataLoader workers are forked after this is built, so they share it."""
    def __init__(self, path):
        import pyarrow.parquet as pq
        self.bytes = {row["path"]: row["bytes"] for row in pq.read_table(path, columns=["image"]).column("image").to_pylist()}
    def load(self, name):
        return Image.open(io.BytesIO(self.bytes[name])).convert("RGB")

class Captions(Dataset):
    """entries: (source, image name, caption) -- or a list of captions, one of which is drawn at random
    each time the image is read (training); `fixed` always takes the first (validation)."""
    def __init__(self, entries, images_by_source, fixed=False):
        self.entries, self.images, self.fixed = entries, images_by_source, fixed
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, i):
        source, name, captions = self.entries[i]
        caption = captions if isinstance(captions, str) else (captions[0] if self.fixed else random.choice(captions))
        return self.images[source].load(name), caption, MODE[source]

def collate(batch):
    enc = processor(text=[chat(c, mode) for _, c, mode in batch], images=[[img] for img, _, _ in batch], return_tensors="pt", padding=True)
    labels = enc["input_ids"].clone()
    k = len(ASSISTANT_IDS)
    for row in range(labels.shape[0]):
        ids = enc["input_ids"][row].tolist()
        start = max(i for i in range(len(ids) - k + 1) if ids[i:i + k] == ASSISTANT_IDS) + k
        labels[row, :start] = -100
    labels[enc["attention_mask"] == 0] = -100
    enc["labels"] = labels
    return enc

class ModeBatches:
    """Batch sampler whose batches hold ONE mode only -- a batch of one-sentence captions padded to a
    paragraph's length wastes compute, and mixing them lets the longer captions dominate the batch's
    loss. Training: the ragged tail of each mode is dropped and the batches are shuffled together,
    differently every epoch. Evaluation: in order, nothing dropped."""
    def __init__(self, modes, batch_size, shuffle):
        self.groups = collections.defaultdict(list)
        for i, mode in enumerate(modes):
            self.groups[mode].append(i)
        self.batch_size, self.shuffle, self.epoch = batch_size, shuffle, 0
    def __len__(self):
        n = self.batch_size
        return sum(len(g) // n if self.shuffle else -(-len(g) // n) for g in self.groups.values())
    def __iter__(self):
        rng = random.Random(SEED * 1000 + self.epoch)
        self.epoch += 1
        n, batches = self.batch_size, []
        for mode in sorted(self.groups):
            idx = list(self.groups[mode])
            if self.shuffle:
                rng.shuffle(idx)
            stop = len(idx) - len(idx) % n if self.shuffle else len(idx)
            batches += [idx[i:i + n] for i in range(0, stop, n)]
        if self.shuffle:
            rng.shuffle(batches)
        return iter(batches)

from peft import LoraConfig, get_peft_model

def add_lora(model):
    """LoRA on the text model's projections only (the regex skips the SigLIP vision layers, which share
    the projection names), plus the vision-to-text connector trained in full."""
    model = get_peft_model(model, LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.05,
        target_modules=r".*text_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
    ))
    for name, p in model.named_parameters():
        if "connector" in name:
            p.requires_grad = True
    return model

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

@torch.no_grad()
def generate(model, entries, images, mode, batch=16, max_new_tokens=None):
    """entries: (image name, reference(s)) pairs; only the names are used."""
    model.eval()
    tokenizer.padding_side = "left"
    prompt = chat(None, mode)
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS[mode]
    texts = []
    for i in range(0, len(entries), batch):
        chunk = entries[i:i + batch]
        inputs = processor(text=[prompt] * len(chunk), images=[[images.load(n)] for n, _ in chunk], return_tensors="pt", padding=True).to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05)
        texts += [t.strip() for t in processor.batch_decode(ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]
    tokenizer.padding_side = "right"
    return texts

def caption_metrics(references, hypotheses):
    """references: one string, or a list of strings, per image."""
    norm = lambda s: " ".join(re.findall(r"[a-z0-9]+", s.lower()))
    refs = [[r] if isinstance(r, str) else list(r) for r in references]
    gts = {i: [norm(x) for x in rs] for i, rs in enumerate(refs)}
    res = {i: [norm(h) or "empty"] for i, h in enumerate(hypotheses)}
    bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
    rouge, _ = Rouge().compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    return {"BLEU-1": bleu[0], "BLEU-2": bleu[1], "BLEU-3": bleu[2], "BLEU-4": bleu[3], "ROUGE-L": rouge, "CIDEr": cider,
            "mean_words": float(np.mean([len(h.split()) for h in hypotheses])),
            "reference_mean_words": float(np.mean([len(x.split()) for rs in refs for x in rs]))}

## 3. Pre-flight: the whole train / save / reload path, before any download

Round one's first Kaggle run spent 20 minutes downloading 12GB and then died at `get_peft_model` on a library-version clash. This cell runs the *same* functions the real run uses -- LoRA injection, two fp16-autocast optimiser steps with gradient scaling on a batch that mixes both prompts, generation in both modes, merge, save and reload of the exported artifact -- on four synthetic images, so an environment problem shows up in the first two minutes instead of after the downloads. It also **finds the largest micro-batch that fits** (one worst-case-length forward+backward per candidate, keeping 15% of GPU memory free, with gradient accumulation making up the difference to an effective batch of 32): round one's second run passed a batch-4 pre-flight and then ran out of memory on its first real step at batch 32.

In [ ]:
class SyntheticImages:
    """Stands in for ZipImages: deterministic random 512px images."""
    def load(self, name):
        rng = np.random.default_rng(sum(map(ord, name)))
        return Image.fromarray(rng.integers(0, 255, (512, 512, 3), dtype=np.uint8))

def preflight():
    t0 = time.time()
    use_amp = DEVICE == "cuda"
    m = add_lora(AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE))
    params = [p for p in m.parameters() if p.requires_grad]
    assert params, "no trainable parameters after add_lora()"
    opt = torch.optim.AdamW(params, lr=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    images = SyntheticImages()
    batch = collate([(images.load(f"x{i}"), f"The image shows {i + 2} large buildings beside a road on the left side.", ("detailed", "brief")[i % 2]) for i in range(4)])
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    m.train()
    for _ in range(2):  # the real loop's exact sequence: autocast forward, scaled backward, unscale, clip, step
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = m(**batch).loss
        assert torch.isfinite(loss), f"non-finite loss in the very first steps ({float(loss)}) -- fp16 is overflowing"
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    for mode in PROMPTS:
        out = generate(m, [("a", ""), ("b", "")], images, mode, batch=2, max_new_tokens=8)
        assert len(out) == 2
    merged = m.merge_and_unload().half()
    tmp = "/tmp/preflight_model"
    merged.save_pretrained(tmp, safe_serialization=True); processor.save_pretrained(tmp)
    del m, merged, opt
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    reloaded = AutoModelForImageTextToText.from_pretrained(tmp, dtype=torch.float16).to(DEVICE)
    for mode in PROMPTS:
        out = generate(reloaded, [("a", "")], images, mode, batch=1, max_new_tokens=8)
        assert len(out) == 1
    del reloaded
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"pre-flight OK in {time.time() - t0:.0f}s: LoRA + autocast train steps, generate (both modes), merge/save/reload all run in THIS environment")

preflight()

def pick_batch_size(target):
    """Round one's second run passed the pre-flight above (batch 4) and then ran out of GPU memory on
    its first real step at batch 32 -- the fp32 logits alone are 32 x ~200 tokens x 49k vocab x 4 B =
    1.2GB. So: try the largest micro-batch first with a WORST-CASE-length batch (144-word detailed
    captions; the longest real one is 142), one full forward+backward, and keep 15% of GPU memory as
    headroom for real-batch variation and allocator fragmentation. Gradient accumulation makes up the
    difference to the target effective batch."""
    global BATCH, ACCUM
    if DEVICE != "cuda":
        return
    m = add_lora(AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE))
    m.train()
    params = [p for p in m.parameters() if p.requires_grad]
    scaler = torch.amp.GradScaler("cuda")
    images = SyntheticImages()
    caption = " ".join(["The image shows a large building near a road on the left side of the frame."] * 12)
    total = torch.cuda.get_device_properties(0).total_memory
    chosen = None
    for cand in [c for c in (32, 16, 8, 4, 2, 1) if c <= target]:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        try:
            batch = {k: v.to(DEVICE) for k, v in collate([(images.load(f"y{i}"), caption, "detailed") for i in range(cand)]).items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = m(**batch).loss
            scaler.scale(loss).backward()
            peak = torch.cuda.max_memory_allocated()
        except torch.cuda.OutOfMemoryError:
            peak = total
        for p in params:
            p.grad = None
        batch = loss = None
        print(f"  micro-batch {cand:2d}: peak {peak / 1e9:5.1f} GB of {total / 1e9:.1f} GB -> {'fits' if peak < 0.85 * total else 'too big'}")
        if peak < 0.85 * total:
            chosen = cand
            break
    assert chosen, "not even a micro-batch of 1 fits"
    BATCH, ACCUM = chosen, max(1, target // chosen)
    del m, params, scaler
    torch.cuda.empty_cache()
    print(f"micro-batch {BATCH} x {ACCUM} accumulation steps = effective batch {BATCH * ACCUM}")

pick_batch_size(TARGET_BATCH)

## 4. Get the data

**VRSBench** (unchanged from round one): `Images_train.zip` (8.4GB) and `Images_val.zip` (4.0GB) are PNG-in-zip, opened in place by the dataset above -- no extraction. `hf_transfer` (parallel chunks, ~150 MB/s on Kaggle) is tried first, with `wget` as the fallback. **NWPU-Captions:** the 20MB captions JSON comes from GitHub raw (with retries) and the 425MB image parquet from the Hugging Face CDN. Scratch space is `/kaggle/temp` if it exists, else `/tmp` -- *not* `/kaggle/working`, whose contents are saved as this run's output.

In [ ]:
HF = "https://huggingface.co/datasets/xiang709/VRSBench/resolve/main/"
FILES = ["VRSBench_train.json", "VRSBench_EVAL_Cap.json", "Images_train.zip", "Images_val.zip"]
NWPU_JSON_URL = "https://raw.githubusercontent.com/HaiyanHuang98/NWPU-Captions/main/dataset_nwpu.json"
NWPU_REPO = "jonathan-roberts1/NWPU-RESISC45"
SCRATCH_DIR = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"

def get_file(name):
    if DATA_ROOT:
        return os.path.join(DATA_ROOT, name)
    dest = os.path.join(SCRATCH_DIR, name)
    if os.path.exists(dest):
        return dest
    t0 = time.time()
    try:  # hf_transfer downloads in parallel chunks; a plain wget of the same file managed ~8 MB/s
        from huggingface_hub import hf_hub_download
        hf_hub_download("xiang709/VRSBench", name, repo_type="dataset", local_dir=SCRATCH_DIR)
        how = "hf_transfer"
    except Exception as exc:
        print(f"  hf download failed for {name} ({type(exc).__name__}: {str(exc)[:100]}) -- falling back to wget")
        subprocess.run(["wget", "-q", "-c", HF + name, "-O", dest], check=True)  # -c: resume a dropped connection
        how = "wget"
    size = os.path.getsize(dest)
    print(f"downloaded {name} ({size / 1e9:.2f} GB) in {time.time() - t0:.0f}s via {how} ({size / 1e6 / max(time.time() - t0, 1):.0f} MB/s)", flush=True)
    return dest

def get_nwpu():
    """-> (images parquet, captions json)"""
    if DATA_ROOT:
        return os.path.join(DATA_ROOT, "nwpu_mini.parquet"), os.path.join(DATA_ROOT, "dataset_nwpu.json")
    import requests
    json_dest = os.path.join(SCRATCH_DIR, "dataset_nwpu.json")
    for attempt in range(5):
        if os.path.exists(json_dest) and os.path.getsize(json_dest) > 20e6:  # the real file is 20.5MB
            break
        try:
            response = requests.get(NWPU_JSON_URL, timeout=120)
            response.raise_for_status()
            with open(json_dest, "wb") as f:
                f.write(response.content)
        except Exception as exc:
            print(f"  captions download attempt {attempt + 1} failed ({type(exc).__name__}: {str(exc)[:80]}); retrying", flush=True)
            time.sleep(5 * (attempt + 1))
    assert os.path.exists(json_dest) and os.path.getsize(json_dest) > 20e6, "could not fetch dataset_nwpu.json from GitHub"
    from huggingface_hub import HfApi, hf_hub_download
    t0 = time.time()
    parquet = next(f for f in HfApi().list_repo_files(NWPU_REPO, repo_type="dataset") if f.startswith("data/") and f.endswith(".parquet"))
    path = hf_hub_download(NWPU_REPO, parquet, repo_type="dataset", local_dir=SCRATCH_DIR)
    print(f"downloaded NWPU-RESISC45 images ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t0:.0f}s; captions {os.path.getsize(json_dest) / 1e6:.1f} MB", flush=True)
    return path, json_dest

if not DATA_ROOT:
    free_gb = shutil.disk_usage(SCRATCH_DIR).free / 1e9
    print(f"free space in {SCRATCH_DIR}: {free_gb:.0f} GB")
    assert free_gb > 16, "need ~13GB for the two image zips plus headroom"

paths = {name: get_file(name) for name in FILES}
for name, p in paths.items():
    print(f"{name:22s} {os.path.getsize(p) / 1e6:10.1f} MB")
nwpu_parquet, nwpu_json = get_nwpu()

## 5. Captions: cleaning, splits and loaders

**VRSBench (detailed):** the cleaning rules were developed against all 20,264 real training captions (rules for Google Earth, the GF/JL satellite names and resolution claims); the `usable` filter then drops the ~8% whose cleaned text is still broken -- 18,324 captions survive after the 300 validation images are held out. The same rules are applied to the evaluation references, so training and evaluation see the same style. The shuffle, the validation split and the 1,000 test images are the same code and seed as round one, so the test set is *the same 1,000 images* and the numbers are directly comparable.

**NWPU-Captions (brief):** captions are normalised (space before punctuation removed, capitalised, full stop) and kept if they have 4-60 words and are ASCII. The authors' own train / val / test split is used; validation and test are stratified over the 45 classes.

In [ ]:
_PROV = r"(?:google\s?earth|gf(?:-?\d+)?|jl(?:-?\d+)?|gaofen(?:-?\d+)?|jilin(?:-?\d+)?)(?:\s+(?:source|satellite|imagery|sensor))?"
_SRC = r"(?:sourced\s+from|taken\s+from|captured\s+(?:by|from)|obtained\s+from|acquired\s+(?:by|from)|provided\s+(?:by|from)|courtesy\s+of|generated\s+by|from|via|on|by)"
_RES = r"(?:medium|high|low|moderate|specific|specified|unspecified|unknown|visible|explicit)[- ]?resolution"
_RULES = [
    # "..., sourced from GoogleEarth, ..." / "captured by GF with medium resolution"
    (re.compile(rf",?\s+(?:which\s+is\s+|that\s+is\s+)?{_SRC}\s+{_PROV}(?:\s+(?:with|at|in|of)\s+(?:a\s+)?{_RES})?\s*,?", re.I), " "),
    (re.compile(rf"\b(?:is|was)\s+{_SRC}\s+{_PROV}\s+and\b", re.I), "and"),
    # "the GoogleEarth image" / "the GF grayscale image"
    (re.compile(rf"\b{_PROV}\s+(?=(?:high-resolution\s+)?(?:aerial\s+|satellite\s+|grayscale\s+|color\s+)?(?:image|view|photo|imagery))", re.I), ""),
    (re.compile(rf"\b{_PROV}\b", re.I), "the imagery"),
    # resolution claims the model could never verify
    (re.compile(rf"\s*,?\s+(?:with|at|in|of)\s+(?:a\s+|an\s+)?{_RES}", re.I), ""),
    (re.compile(rf"\b{_RES}\s+(?=(?:aerial\s+|satellite\s+|grayscale\s+|color\s+|overhead\s+)?(?:image|view|photo|imagery|picture))", re.I), ""),
    (re.compile(r"\b(?:high|medium|low|moderate)[- ]resolution\s+(?=[a-z])", re.I), ""),
    (re.compile(r"\s+without\s+(?:a\s+)?(?:specified|stated|given|provided|specific)\s+resolution", re.I), ""),
    (re.compile(r"\s+with\s+(?:an?\s+)?(?:unspecified|unknown|no)\s+resolution", re.I), ""),
]

def clean_caption(text):
    out = text
    for rx, repl in _RULES:
        out = rx.sub(repl, out)
    out = re.sub(r"\s+([,.;:])", r"\1", out)
    out = re.sub(r",\s*,", ",", out)
    out = re.sub(r",\s*\.", ".", out)
    out = re.sub(r"\s{2,}", " ", out).strip()
    out = re.sub(r"\b([Aa]) (?=(?:aerial|overhead|urban|industrial|open|angled|oblique|elevated|image)\b)", lambda m: m.group(1) + "n ", out)
    out = re.sub(r"^(The|This) image,\s+(with|features|shows|displays|depicts|captures|showcases|presents|contains|includes)\b", lambda m: f"{m.group(1)} image {m.group(2)}", out)
    return out[:1].upper() + out[1:] if out else out

_BROKEN = re.compile(r"goog|sourced|provided by|courtesy|\bGF\b|\bJL\b|gaofen|jilin|the imagery|resolution|^(The|This) image (and|is|was|with|by)\b|\bby the\b\s*[.,]", re.I)
def usable(caption):
    return len(caption.split()) >= 12 and not _BROKEN.search(caption)

def clean_scene_caption(text):
    t = re.sub(r"\s+", " ", text).strip()
    t = re.sub(r"\s+([,.;:!?])", r"\1", t)  # "... on the open area ." -> "... on the open area."
    t = re.sub(r"([.!?])\1+", r"\1", t)
    if t and t[-1] not in ".!?":
        t += "."
    return t[:1].upper() + t[1:]

def usable_scene(caption):
    return 4 <= len(caption.split()) <= 60 and caption.isascii()

# ---- VRSBench (detailed) ----
train_json = json.load(open(paths["VRSBench_train.json"]))
raw_captions = {it["image"]: it["conversations"][1]["value"] for it in train_json
                if it["conversations"][0]["value"].startswith("<image>\n[caption]")}
with zipfile.ZipFile(paths["Images_train.zip"]) as z:
    train_members = set(z.namelist())
pairs = [(n, clean_caption(c)) for n, c in raw_captions.items() if f"Images_train/{n}" in train_members]
n_before = len(pairs)
pairs = [(n, c) for n, c in pairs if usable(c)]
print(f"VRSBench: {len(raw_captions)} caption images, {n_before} with an image in the zip, {n_before - len(pairs)} dropped as still-broken after cleaning")
random.Random(SEED).shuffle(pairs)
vrs_val = [("vrs_train", n, c) for n, c in pairs[:N_VAL]]
vrs_train = [("vrs_train", n, c) for n, c in pairs[N_VAL:]]
if SMOKE_TEST:
    vrs_train = vrs_train[:16]

eval_json = json.load(open(paths["VRSBench_EVAL_Cap.json"]))
with zipfile.ZipFile(paths["Images_val.zip"]) as z:
    eval_members = set(z.namelist())
test_pool = [(x["image_id"], clean_caption(x["ground_truth"])) for x in eval_json if f"Images_val/{x['image_id']}" in eval_members]
test_pool = [(n, c) for n, c in test_pool if usable(c)]  # the same still-broken filter as the training captions
vrs_test = random.Random(SEED).sample(test_pool, min(N_TEST, len(test_pool)))  # (name, reference) pairs

lens = [len(c.split()) for _, _, c in vrs_train]
print(f"VRSBench (detailed): train {len(vrs_train)} | val {len(vrs_val)} | test {len(vrs_test)} (of {len(test_pool)} eval images); train caption length mean {np.mean(lens):.1f} words, max {max(lens)}")
for _, n, c in vrs_train[:2]:
    print(" -", n, "|", c[:160])

# ---- NWPU-Captions (brief) ----
nwpu_images = ParquetImages(nwpu_parquet)
have = set(nwpu_images.bytes)
nwpu_data = json.load(open(nwpu_json))
scene_items, dropped_captions = [], 0
for cls, items in nwpu_data.items():
    for it in items:
        if it["filename"] not in have:
            continue
        raw = [it[k] for k in ("raw", "raw_1", "raw_2", "raw_3", "raw_4") if it.get(k)]
        caps = [c for c in (clean_scene_caption(r) for r in raw) if usable_scene(c)]
        dropped_captions += len(raw) - len(caps)
        if caps:
            scene_items.append({"name": it["filename"], "cls": cls, "split": it["split"], "caps": caps})
joined = len(scene_items) / len(have)
print(f"NWPU-Captions: {len(have)} images in the mirror, {len(scene_items)} with usable captions ({joined:.1%}); {dropped_captions} individual captions dropped by the filter")
assert joined >= 0.995, "the captions and the images do not line up (filenames differ?) -- see the join above"

scene_rng = random.Random(SEED)
def stratified(items, per_class):
    by_class = collections.defaultdict(list)
    for x in items:
        by_class[x["cls"]].append(x)
    out = []
    for cls in sorted(by_class):
        pool = sorted(by_class[cls], key=lambda x: x["name"])
        out += scene_rng.sample(pool, min(per_class, len(pool)))
    return out

by_split = collections.defaultdict(list)
for x in scene_items:
    by_split[x["split"]].append(x)
nwpu_train = [("nwpu", x["name"], x["caps"]) for x in by_split["train"]]
if SMOKE_TEST:
    nwpu_train = nwpu_train[:16]
nwpu_val = [("nwpu", x["name"], x["caps"][0]) for x in stratified(by_split["val"], PER_CLASS_VAL)]
nwpu_test_items = stratified(by_split["test"], PER_CLASS_TEST)
nwpu_test = [(x["name"], x["caps"]) for x in nwpu_test_items]  # (name, ALL references)
nwpu_class_of = {x["name"]: x["cls"] for x in scene_items}
assert nwpu_train and nwpu_val and nwpu_test, "an NWPU split came out empty"
lens = [len(c.split()) for _, _, cs in nwpu_train for c in cs]
print(f"NWPU (brief): train {len(nwpu_train)} images | val {len(nwpu_val)} | test {len(nwpu_test)} ({len({x['cls'] for x in nwpu_test_items})} classes); train caption length mean {np.mean(lens):.1f} words")
for _, n, cs in nwpu_train[:3]:
    print(" -", n, "|", cs[0])

train_images = ZipImages(paths["Images_train.zip"], "Images_train")
eval_images = ZipImages(paths["Images_val.zip"], "Images_val")
IMAGES = {"vrs_train": train_images, "vrs_eval": eval_images, "nwpu": nwpu_images}

def make_loader(entries, shuffle):
    modes = [MODE[e[0]] for e in entries]
    return DataLoader(Captions(entries, IMAGES, fixed=not shuffle), batch_sampler=ModeBatches(modes, BATCH, shuffle),
                      collate_fn=collate, num_workers=WORKERS, persistent_workers=(WORKERS > 0))

train_loader = make_loader(vrs_train + nwpu_train, shuffle=True)
val_loader_detailed = make_loader(vrs_val, shuffle=False)
val_loader_brief = make_loader(nwpu_val, shuffle=False)
print(f"one epoch = {len(train_loader)} micro-batches of {BATCH} ({len(vrs_train) // BATCH} detailed + {len(nwpu_train) // BATCH} brief)")

for mode, loader, lo, hi in (("detailed", val_loader_detailed, 20, 250), ("brief", val_loader_brief, 4, 120)):
    probe = next(iter(loader))
    supervised = int((probe["labels"] != -100).sum(1).float().mean())
    first = probe["labels"][0]
    print(f"[{mode}] batch tensors: input_ids {tuple(probe['input_ids'].shape)}, pixel_values {tuple(probe['pixel_values'].shape)}; supervised tokens per caption ~{supervised}")
    print(f"[{mode}] supervised text of the first caption:", tokenizer.decode(first[first != -100])[:160])
    assert lo < supervised < hi, f"answer-only masking looks wrong for the {mode} mode"

## 6. Round one's numbers, for comparison

Round one measured the un-tuned base model and the VRSBench-only fine-tune on these exact 1,000 VRSBench test images (kernel `satquery-caption-vrsbench` v3). They are recorded here rather than re-measured -- the base model's 1,000 generations take ten minutes of GPU and cannot change. There is no zero-shot baseline for the brief mode: an un-tuned model writes a paragraph for any prompt, so it would only measure how badly it ignores the instruction.

In [ ]:
PREV_RUN = {
    "zero_shot": {"BLEU-1": 0.199, "BLEU-2": 0.096, "BLEU-3": 0.046, "BLEU-4": 0.022, "ROUGE-L": 0.199, "CIDEr": 0.001, "mean_words": 106.3},
    "vrsbench_only": {"BLEU-1": 0.458, "BLEU-2": 0.288, "BLEU-3": 0.184, "BLEU-4": 0.118, "ROUGE-L": 0.337, "CIDEr": 0.300, "mean_words": 45.9},
}

def evaluate(model, tag, entries, images, mode):
    t0 = time.time()
    hyps = generate(model, entries, images, mode)
    scores = caption_metrics([c for _, c in entries], hyps)
    print(f"\n=== {tag}: {len(hyps)} test images, {time.time() - t0:.0f}s ===")
    print("  " + "  ".join(f"{k} {v:.3f}" for k, v in scores.items()))
    return scores, hyps

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters (fp32 master weights; fp16 autocast does the fast math)")

## 7. LoRA on the language model + a trainable connector

The vision encoder stays frozen. LoRA (r=32) goes on every attention and MLP projection of the *text* model only (the regex excludes the SigLIP vision layers, whose projections share the same names); the small connector that maps vision features into the language model's space is trained in full, at a quarter of the learning rate.

In [ ]:
model = add_lora(model)
lora_params = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
connector_params = [p for n, p in model.named_parameters() if p.requires_grad and "connector" in n]
total = sum(p.numel() for p in model.parameters())
print(f"trainable: LoRA {sum(p.numel() for p in lora_params) / 1e6:.1f}M + connector {sum(p.numel() for p in connector_params) / 1e6:.1f}M of {total / 1e6:.0f}M total")
assert lora_params and connector_params
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 8. Train

AdamW, warm-up then cosine decay, fp16 autocast + gradient scaling on GPU. Once per epoch the validation loss is measured on each mode's held-out images (teacher-forced) and the weights with the lowest **mean of the two** are kept and saved to `best_trainable.pt` (~120MB, so a failed export can be redone without retraining). A non-finite loss skips the step, and more than 5% of them aborts the run -- an fp16 overflow should stop the run early, not quietly poison it.

In [ ]:
optimizer = torch.optim.AdamW(
    [{"params": lora_params, "lr": LR}, {"params": connector_params, "lr": LR / 4}], weight_decay=0.01)
steps_per_epoch = len(train_loader) // ACCUM  # optimiser steps (each is ACCUM micro-batches)
steps_total = EPOCHS * steps_per_epoch
warmup = max(1, int(0.03 * steps_total))
def lr_factor(step):
    if step < warmup:
        return (step + 1) / warmup
    progress = (step - warmup) / max(1, steps_total - warmup)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def validation_loss(loader):
    model.eval()
    total, count = 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            n = int((batch["labels"] != -100).sum())
            total += float(model(**batch).loss) * n
        count += n
    return total / max(count, 1)

def trainable_state():
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

CKPT_DIR = "/tmp/caption_ckpt" if SMOKE_TEST else "/kaggle/working/caption_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
best_val, best_state, skipped, step = float("inf"), None, 0, 0
val_history = []
t_start = time.time()
print(f"{steps_total} optimiser steps ({steps_per_epoch} per epoch), micro-batch {BATCH} x {ACCUM} = effective batch {BATCH * ACCUM}")
for epoch in range(EPOCHS):
    model.train()
    running, n_micro, t_epoch = 0.0, 0, time.time()
    optimizer.zero_grad(set_to_none=True)
    for i, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = model(**batch).loss
        if not torch.isfinite(loss):  # drops the partial accumulation window too -- rare, and safer than stepping on it
            skipped += 1
            optimizer.zero_grad(set_to_none=True)
            assert skipped <= max(3, 0.05 * (epoch * len(train_loader) + i + 1)), "too many non-finite losses -- fp16 is overflowing"
            continue
        scaler.scale(loss / ACCUM).backward()
        running += loss.item(); n_micro += 1
        if (i + 1) % ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(lora_params + connector_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            if step % 100 == 0:
                print(f"  step {step}/{steps_total}  loss {running / max(n_micro, 1):.4f}  ({time.time() - t_start:.0f}s)", flush=True)
    val_d, val_b = validation_loss(val_loader_detailed), validation_loss(val_loader_brief)
    val = (val_d + val_b) / 2
    val_history.append({"epoch": epoch + 1, "val_detailed": val_d, "val_brief": val_b, "val_mean": val})
    marker = ""
    if val < best_val:
        best_val, best_state, marker = val, trainable_state(), "  <- best"
        torch.save(best_state, os.path.join(CKPT_DIR, "best_trainable.pt"))
    print(f"epoch {epoch + 1}/{EPOCHS}  train loss {running / max(n_micro, 1):.4f}  val loss: detailed {val_d:.4f}  brief {val_b:.4f}  mean {val:.4f}  ({time.time() - t_epoch:.0f}s){marker}", flush=True)
    for mode, ents, imgs in (("detailed", [(n, c) for _, n, c in vrs_val[:2]], train_images), ("brief", [(n, c) for _, n, c in nwpu_val[:3]], nwpu_images)):
        for (name, ref), hyp in zip(ents, generate(model, ents, imgs, mode)):
            print(f"    [{mode} {name}]\n      REF: {ref[:160]}\n      GEN: {hyp[:200]}")
print(f"\ntraining took {(time.time() - t_start) / 60:.1f} min; best mean val loss {best_val:.4f}; skipped {skipped} non-finite steps")
model.load_state_dict(best_state, strict=False)

## 9. Test

**Detailed mode** on the same 1,000 VRSBench test images as round one: it must not have regressed, and the un-tuned base model and round one's fine-tune are the reference points. **Brief mode** on 30 test images per scene class (five references each), plus whether the caption *names its own scene*: for each class, the words its training captions use far more than other classes' do (a word must appear in 25% of the class's captions and be 3x more frequent there than elsewhere; top 6) -- the fraction of generated captions that use at least one of them, next to the same fraction for the human references (the ceiling, since a caption can be right and worded differently). The brief evaluation is wrapped so a bug in it cannot cost the export below.

In [ ]:
detailed_scores, detailed_hyps = evaluate(model, "DETAILED mode, VRSBench test", vrs_test, eval_images, "detailed")
print("\ndetailed mode: un-tuned base -> round one (VRSBench only) -> this run")
for k in ("BLEU-4", "ROUGE-L", "CIDEr", "mean_words"):
    print(f"  {k:12s} {PREV_RUN['zero_shot'][k]:8.3f} -> {PREV_RUN['vrsbench_only'][k]:8.3f} -> {detailed_scores[k]:8.3f}")
for (name, ref), hyp in list(zip(vrs_test, detailed_hyps))[:3]:
    print(f"  [{name}]\n    REF: {ref[:230]}\n    GEN: {hyp[:230]}")

STOPWORDS = {"the", "and", "are", "has", "have", "its", "with", "some", "there", "that", "this", "which", "from", "for", "was", "were", "into", "also", "very", "one", "two", "many", "lots", "next", "picture", "image"}
brief_scores, class_vocab_rates, brief_hyps = {}, {}, []
try:
    brief_scores, brief_hyps = evaluate(model, "BRIEF mode, NWPU-Captions test", nwpu_test, nwpu_images, "brief")

    train_by_class = collections.defaultdict(list)
    for x in by_split["train"]:
        train_by_class[x["cls"]].append(x["caps"])
    n_all = sum(len(v) for v in train_by_class.values())
    doc_freq = collections.Counter()
    class_freq = {}
    for cls, caps_list in train_by_class.items():
        class_freq[cls] = collections.Counter(w for caps in caps_list for w in set(re.findall(r"[a-z]+", " ".join(caps).lower())))
        doc_freq.update(class_freq[cls])
    vocab = {}
    for cls, freq in class_freq.items():
        n_cls = len(train_by_class[cls])
        scored = []
        for w, k in freq.items():
            p_in, p_out = k / n_cls, (doc_freq[w] - k) / max(1, n_all - n_cls)
            if len(w) >= 3 and w not in STOPWORDS and p_in >= 0.25 and p_in > 3 * p_out:
                scored.append((p_in - p_out, w))
        vocab[cls] = {w for _, w in sorted(scored, reverse=True)[:6]}
    names_scene = lambda cls, text: bool(vocab[cls] & set(re.findall(r"[a-z]+", text.lower())))

    per_class = collections.defaultdict(lambda: [0, 0, 0, 0])  # model hits, model n, reference hits, reference n
    for (name, refs), hyp in zip(nwpu_test, brief_hyps):
        cls = nwpu_class_of[name]
        if not vocab[cls]:
            continue
        row = per_class[cls]
        row[0] += names_scene(cls, hyp); row[1] += 1
        row[2] += sum(names_scene(cls, r) for r in refs); row[3] += len(refs)
    model_rate = sum(r[0] for r in per_class.values()) / max(1, sum(r[1] for r in per_class.values()))
    human_rate = sum(r[2] for r in per_class.values()) / max(1, sum(r[3] for r in per_class.values()))
    class_vocab_rates = {c: {"model": r[0] / r[1], "human_reference": r[2] / r[3]} for c, r in per_class.items()}
    print(f"\nbrief mode names its own scene: {model_rate:.1%} of generated captions (human references: {human_rate:.1%}) over {len(per_class)} classes")
    for c, r in sorted(class_vocab_rates.items(), key=lambda kv: kv[1]["model"])[:14]:
        print(f"  weakest: {c:24s} model {r['model']:5.0%}  references {r['human_reference']:5.0%}   class words: {sorted(vocab[c])}")
    brief_scores["names_own_scene"] = model_rate
    brief_scores["names_own_scene_human_reference"] = human_rate
    print("\nbrief captions, first test image of each of 14 classes:")
    shown = set()
    for (name, refs), hyp in zip(nwpu_test, brief_hyps):
        cls = nwpu_class_of[name]
        if cls not in shown and len(shown) < 14:
            shown.add(cls)
            print(f"  [{cls}] GEN: {hyp}\n            REF: {refs[0]}")
except Exception:
    import traceback
    traceback.print_exc()
    print("!! brief-mode evaluation failed -- continuing to the export (best_trainable.pt is already saved)")

generations = {
    "detailed": [{"name": n, "reference": r, "generated": h} for (n, r), h in zip(vrs_test, detailed_hyps)],
    "brief": [{"name": n, "class": nwpu_class_of[n], "references": refs, "generated": h} for (n, refs), h in zip(nwpu_test, brief_hyps)],
}
json.dump(generations, open(os.path.join(CKPT_DIR, "test_generations.json"), "w"))  # ~0.5MB, next to best_trainable.pt
print(f"saved {len(generations['detailed'])} detailed + {len(generations['brief'])} brief test generations to {CKPT_DIR}/test_generations.json")

## 10. Export

Merge the LoRA into the base weights and save a plain half-precision checkpoint directory (about 1GB) that `models/captioning/caption_tool.py` loads with `AutoModelForImageTextToText.from_pretrained` -- no PEFT needed at inference. `caption_meta.json` carries both prompts and both modes' metrics. Then reload the *saved artifact* and caption a test image in each mode, so a broken export fails here and not on the laptop.

In [ ]:
OUT_DIR = "/tmp/caption_model" if SMOKE_TEST else "/kaggle/working/caption_model"
os.makedirs(OUT_DIR, exist_ok=True)

merged = model.merge_and_unload().half()
merged.save_pretrained(OUT_DIR, safe_serialization=True)
processor.save_pretrained(OUT_DIR)
meta = {
    "base_model": MODEL_ID,
    "prompt": PROMPTS["detailed"],  # what round one's readers use: the detailed prompt
    "prompts": PROMPTS,
    "do_image_splitting": False,
    "task": "single-image captioning in two modes: 'detailed' (VRSBench, an object-centric paragraph) and 'brief' (NWPU-Captions, a one-sentence scene description that covers land cover)",
    "train_images": {"detailed": len(vrs_train), "brief": len(nwpu_train)}, "epochs": EPOCHS,
    "best_val_loss": best_val, "val_history": val_history,
    "detailed": {"test_images": len(vrs_test), "zero_shot_metrics": PREV_RUN["zero_shot"], "round_one_metrics": PREV_RUN["vrsbench_only"], "fine_tuned_metrics": detailed_scores},
    "brief": {"test_images": len(nwpu_test), "fine_tuned_metrics": brief_scores, "class_vocabulary_rates": class_vocab_rates},
    "zero_shot_metrics": PREV_RUN["zero_shot"], "fine_tuned_metrics": detailed_scores,  # round one's keys, kept for older readers
}
json.dump(meta, open(os.path.join(OUT_DIR, "caption_meta.json"), "w"), indent=1)
size_mb = sum(os.path.getsize(os.path.join(OUT_DIR, f)) for f in os.listdir(OUT_DIR)) / 1e6
print(f"Exported to {OUT_DIR} ({size_mb:.0f} MB): {sorted(os.listdir(OUT_DIR))}")

del merged, model
torch.cuda.empty_cache() if DEVICE == "cuda" else None
reloaded = AutoModelForImageTextToText.from_pretrained(OUT_DIR, dtype=torch.float16).to(DEVICE)
for mode, entries, images in (("detailed", vrs_test[:2], eval_images), ("brief", nwpu_test[:2], nwpu_images)):
    check = generate(reloaded, entries, images, mode, batch=2)
    assert all(len(t.split()) >= 3 for t in check), f"reloaded artifact produced junk in {mode} mode: {check}"
    print(f"artifact round-trip OK ({mode}):", check[0][:160])